# Data Collection and Processing
This notebook is used to collect agent-based data from all the runs, and validate the runs.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
import polars as pl

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))
from src.utils import move_invalid_runs


from src.analysis.loaders import load_adjacency_matrix, load_agents_data, load_runs_metadata, load_run_data

In [2]:
ROLE_TO_IDX = {
    "Strategic Planner": 0,
    "Language Mediator": 1,
    "Online Friend": 2,
    "Software Engineer": 3,
    "Academic Scholar": 4,
    "Chemist": 5,
    "Human Participant": 6,
    "LLM": 7,
    "Cybersecurity Analyst": 8,
    "Mathematician": 9,
    "Storyteller": 10,
    "Clinical Physician": 11,
    "Biomedical Researcher": 12,
    "Financial Analyst": 13,
    "Lexicographer": 14,
    "Assistant": 15,
}

MATCHES = [
    ("llama-doc", "Clinical Physician"),
    ("llama-base", "LLM"),
    ("llama-base", "Human Participant"),
    ("llama-assistant", "Assistant"),
    ("llama-biomed", "Biomedical Researcher"),
    ("llama-chemist", "Chemist"),
    ("llama-coder", "Software Engineer"),
    ("llama-cyber", "Cybersecurity Analyst"),
    ("llama-finance", "Financial Analyst"),
    ("llama-hermes", "Strategic Planner"),
    ("llama-openmath", "Mathematician"),
    ("llama-lexicographer", "Lexicographer"),
    ("llama-linguist", "Language Mediator"),
    ("llama-roleplay", "Storyteller"),
    ("llama-scholar", "Academic Scholar"),
    ("llama-user", "Online Friend"),
]

## Load Data

In [3]:
# Load all runs

df_total = pl.from_pandas(load_runs_metadata(Path.cwd().parent / "data"/"outputs" / "runs"))

# Filter for completed and validated runs
df_runs = df_total.filter(pl.col('validated') & pl.col('completed')).to_pandas()

print(f"Total validated runs: {len(df_runs)}")
print("\nRuns by setting:")
print(df_runs.groupby("setting").size())
print("\nRuns by graph type:")
print(df_runs.groupby("graph_type").size())

result_path = Path.cwd().parent / "data" / "outputs" / "aggregated"
result_path.mkdir(parents=True, exist_ok=True)

Total validated runs: 1280

Runs by setting:
setting
base_llms         320
experts           320
random_experts    320
random_roles      320
dtype: int64

Runs by graph type:
graph_type
erdos-renyi       640
watts-strogatz    640
dtype: int64


In [4]:
print("Check run distribution across settings and statements:")
df_runs.groupby(["setting","statement_id"]).agg("size").unstack(fill_value=0)

Check run distribution across settings and statements:


statement_id,false_0,false_1,false_2,false_3,false_4,false_5,false_6,false_7,false_8,false_9,true_0,true_1,true_2,true_3,true_4,true_5,true_6,true_7,true_8,true_9
setting,,,,,,,,,,,,,,,,,,,,
base_llms,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16
experts,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16
random_experts,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16
random_roles,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16


In [5]:
# Preview what would be moved (dry run)
stats = move_invalid_runs(
    runs_dir=Path.cwd().parent / "data" / "outputs" / "runs",
    incomplete_dir=Path.cwd().parent / "data" / "outputs" / "incomplete_runs",
    dry_run=False
)

try:
    print(f"\nSummary:")
    print(f"  Incomplete runs: {stats['incomplete']}")
    print(f"  Unvalidated runs: {stats['unvalidated']}")
    print(f"  Total to move: {stats['total']}")
except Exception as e:
    print(f"Error summarizing stats: {e}")
# To actually move the files, run with dry_run=False:
# stats = move_invalid_runs(dry_run=False)


Summary:
  Incomplete runs: 0
  Unvalidated runs: 0
  Total to move: 0


## 1. Processing (Agent-based)

In [6]:
import numpy as np

def compute_stats(belief_con: np.ndarray, y_true: int) -> dict:
    """
    Compute additional statistics from per-round 3-class probability distributions.
    
    Parameters
    ----------
    belief_con : np.ndarray
        Array of shape (T+1, 3) containing [P(true), P(false), P(neither)] for each round.
    y_true : int
        The true label for the statement (0=True, 1=False, 2=Neither).
    
    Returns
    -------
    dict with keys:
        plasticity_tv     : total-variation plasticity (primary outcome, in [0, 1])
        plasticity_true   : per-class plasticity on P(true)
        plasticity_false  : per-class plasticity on P(false)
        plasticity_neither: per-class plasticity on P(neither)
        plasticity_tv_1:  : total-variation plasticity excluding 0th round (primary outcome, in [0, 1])
        net_change        : net distance between final and initial distribution
        net_drift         : net change in probability of the ground_true label
        monotonicity      : net_change/ (T * plasticity_tv), in [0, 1]
                            close to 1 = monotone movement, close to 0 = oscillation
        T                 : number of transitions used
        total_movement    : total movement across rounds in the probability of the correct label
        progress_ratio    : (forward - backward) / total_movement, in [-1, 1]
                            close to 1 = mostly forward, close to -1 = mostly backward
        agent_accuracy    : probability of the true label at the final round
        agent_error       : probability of the false label at the final round
        progress_forward  : total forward movement in probability of the correct label
        progress_backward : total backward movement in probability of the correct label
        progress_efficiency: net_gain / max_possible_gain, in [0, 1]
                            close to 1 = efficient progress, close to 0 = inefficient progress
                            
    """
    if y_true == -1:
        y_true = 2 # do not really happen in our case
    
    P = np.array(belief_con, dtype=float)  # (T+1, 3)
    
    T = P.shape[0] - 1
    diffs = np.abs(np.diff(P, axis=0))         # (T, 3)
    
    plasticity_per_class = diffs.mean(axis=0)  # (3,)
    plasticity_tv = 0.5 * diffs.sum(axis=1).mean()
    plasticity_tv_1 = 0.5 * diffs[1:].sum(axis=1).mean()  # Exclude 0th round
    
    
    net_change = 0.5 * np.abs(P[-1] - P[0]).sum()
    denom = T * plasticity_tv
    monotonicity = float(net_change / denom) if denom > 1e-12 else np.nan
    
    
    net_uncertain =  0.5 * np.abs(P[-1] - P[0]).sum()  # Changes in "neither" probability
    denom_uncertainty = plasticity_per_class[2].mean() * T
    monotonicity_uncertainty = float(net_uncertain / denom_uncertainty)
    
    # Movement in respect to correct label    
    P_correct = P[:, y_true]
    p0, pT = P_correct[0], P_correct[-1]
    net_gain = pT - p0

    eps = 1e-6
    if p0 > 1 - eps or p0 < eps:
       p0 = np.clip(p0, eps, 1 - eps)  # Avoid division by zero or near-zero
    if net_gain >= 0:
        progress_efficiency = net_gain / (1 - p0)   # fraction of upside captured
    else:
        progress_efficiency = net_gain / p0          # fraction of downside realized (negative)
        
        


    
    
    
    diffs_correct = np.diff(P_correct)
    forward = diffs_correct[diffs_correct > 0].sum()    # total movement toward truth
    backward = -diffs_correct[diffs_correct < 0].sum()  # total movement away from truth
    total_movement = forward + backward
    progress_ratio = (forward - backward) / total_movement if total_movement > 0 else 0
    # max_possible_gain = 1.0 - p0    # theoretical ceiling

    # progress_efficiency = net_gain / max_possible_gain if max_possible_gain > 0 else 0
    
    
    ## Accuracy-like scores
    y_prob = P[T, y_true]  # Probability of the true label across rounds    


    return {
        "plasticity_tv":      float(plasticity_tv),
        "plasticity_true":    float(plasticity_per_class[0]),
        "plasticity_tv_1":   float(plasticity_tv_1),
        "plasticity_false":   float(plasticity_per_class[1]),
        "plasticity_neither": float(plasticity_per_class[2]),
        "net_change":         float(net_change),
        "net_gain":           float(net_gain),
        # "net_drift":          float(net_drift),
        "monotonicity":       monotonicity,
        "total_movement":     float(total_movement),
        "progress_ratio":     float(progress_ratio),
        "agent_accuracy":     float(y_prob),
        "agent_error":        float(1 - y_prob),
        "progress_forward": float(forward),
        "progress_backward": float(backward),
        "progress_efficiency": float(progress_efficiency),
        
        "diff_uncertain": float(net_uncertain),
        "monotonicity_uncertainty": float(monotonicity_uncertainty),
        "T":                  T,
    }

In [7]:
import numpy as np

def compute_influence_metrics(belief_ful: np.ndarray, A: np.ndarray) -> dict:
    """
    Per-agent influence metrics from belief trajectories.

    Computes two complementary families of influence:

    Magnitude-based (matches CoevolveSim Eq. 8 from the paper):
        out_influence_mag[i] = mean over (t, j ∈ N(i)) of |Δb_j(t+1)|
            "How much do my neighbors move at t+1?"
        in_influence_mag[i]  = mean over (t, j ∈ N(i)) of |Δb_i(t+1)|
            "How much do I move at t+1?" (plasticity-like)

    Directional (activity-weighted cosine similarity between consecutive belief changes):
        out_influence_dir[i] = weighted mean over (t, j ∈ N(i)) of cos(Δb_i(t), Δb_j(t+1)),
            weighted by ||Δb_i(t)|| · ||Δb_j(t+1)||
            "Do my neighbors move in the same direction as I did?"
        in_influence_dir[i]  = weighted mean over (t, j ∈ N(i)) of cos(Δb_j(t), Δb_i(t+1)),
            weighted by ||Δb_j(t)|| · ||Δb_i(t+1)||
            "Do I move in the same direction as my neighbors did?"

        Weighting by movement magnitude downweights low-activity (near-zero Δb) pairs
        whose cosine is noise-dominated, giving a more reliable estimate in settings
        where many agents barely move.

    Parameters
    ----------
    belief_ful : np.ndarray, shape (N, T+1, 3)
        Per-agent belief trajectories. Last dim is [P(true), P(false), P(neither)].
    A : np.ndarray, shape (N, N)
        Adjacency matrix (binary).

    Returns
    -------
    dict with keys:
        out_influence_mag : (N,) magnitude of neighbors' subsequent updates
        in_influence_mag  : (N,) magnitude of agent's own subsequent update
        out_influence_dir : (N,) activity-weighted directional alignment, neighbors with self
        in_influence_dir  : (N,) activity-weighted directional alignment, self with neighbors
        n_pairs           : (N,) number of valid (j, t) pairs per agent (diagnostic)
        effective_n_out   : (N,) effective sample size for out_dir
                            = (sum of weights)^2 / sum of squared weights (Kish formula)
        effective_n_in    : (N,) effective sample size for in_dir
    """
    N, Tp1, _ = belief_ful.shape
    T = Tp1 - 1

    if T < 2:
        raise ValueError(f"Need at least 2 transitions for influence; got T={T}.")

    delta = np.diff(belief_ful, axis=1)              # (N, T, 3)
    norms = np.linalg.norm(delta, axis=2)            # (N, T)
    tv_mag = 0.5 * np.abs(delta).sum(axis=2)         # (N, T) per-round TV distance
    eps = 1e-12

    out_mag = np.zeros(N)
    in_mag = np.zeros(N)
    out_dir = np.zeros(N)
    in_dir = np.zeros(N)
    n_pairs = np.zeros(N, dtype=int)
    eff_n_out = np.zeros(N)
    eff_n_in = np.zeros(N)

    for i in range(N):
        neighbors = np.where(A[i] > 0)[0]
        if len(neighbors) == 0:
            continue

        # --- Slices ---
        di_lag  = delta[i, :-1]                      # (T-1, 3)
        di_lead = delta[i, 1:]                       # (T-1, 3)
        ni_lag  = norms[i, :-1]                      # (T-1,)
        ni_lead = norms[i, 1:]                       # (T-1,)

        dj_lag  = delta[neighbors, :-1]              # (k, T-1, 3)
        dj_lead = delta[neighbors, 1:]               # (k, T-1, 3)
        nj_lag  = norms[neighbors, :-1]              # (k, T-1)
        nj_lead = norms[neighbors, 1:]               # (k, T-1)

        tv_i_lead = tv_mag[i, 1:]                    # (T-1,)
        tv_j_lead = tv_mag[neighbors, 1:]            # (k, T-1)

        # --- Magnitude metrics ---
        out_mag[i] = tv_j_lead.mean()
        in_mag[i] = tv_i_lead.mean()

        # --- Directional metrics: activity-weighted cosine ---

        # out: cos(Δb_i(t), Δb_j(t+1)), weighted by ||Δb_i(t)|| · ||Δb_j(t+1)||
        dots_out = (di_lag[None] * dj_lead).sum(axis=2)         # (k, T-1)
        weights_out = ni_lag[None] * nj_lead                    # (k, T-1)
        # cosine = dot / (norm_i * norm_j); weighted-mean cosine
        # simplifies: sum(cos * weight) / sum(weight) = sum(dot) / sum(weight)
        # because cos = dot / weight, so cos * weight = dot.
        total_weight_out = weights_out.sum()
        if total_weight_out > eps:
            out_dir[i] = dots_out.sum() / total_weight_out
            # Kish effective sample size
            sumsq_out = (weights_out ** 2).sum()
            if sumsq_out > eps:
                eff_n_out[i] = (total_weight_out ** 2) / sumsq_out

        # in: cos(Δb_j(t), Δb_i(t+1)), weighted by ||Δb_j(t)|| · ||Δb_i(t+1)||
        dots_in = (dj_lag * di_lead[None]).sum(axis=2)          # (k, T-1)
        weights_in = nj_lag * ni_lead[None]                     # (k, T-1)
        total_weight_in = weights_in.sum()
        if total_weight_in > eps:
            in_dir[i] = dots_in.sum() / total_weight_in
            sumsq_in = (weights_in ** 2).sum()
            if sumsq_in > eps:
                eff_n_in[i] = (total_weight_in ** 2) / sumsq_in

        n_pairs[i] = len(neighbors) * (T - 1)

    return {
        "out_influence_mag": out_mag,
        "in_influence_mag":  in_mag,
        "out_influence_dir": out_dir,
        "in_influence_dir":  in_dir,
        "n_pairs":           n_pairs,
        "effective_n_out":   eff_n_out,
        "effective_n_in":    eff_n_in,
    }

In [10]:
recs = []

lf = pl.from_pandas(df_runs)

for run in lf.filter(
    pl.col("setting").is_in(["base_llms", "random_roles", "random_experts", "experts"]) 
).iter_rows(named=True):
    
    run_agents = load_agents_data(run["run_path"])
    graph_data = load_run_data(run["run_path"])
    A, edges = load_adjacency_matrix(run["run_path"])
    
    seed_map = graph_data['config']['network']['seed_map']  # type: ignore
    belief_ful = run_agents["belief_ful"]  # (N_agents, N_rounds, 3)
    belief_cat = run_agents["belief_cat"]  # (N_agents, N_rounds)
    _models = run_agents["models"]
    _roles = run_agents["roles"]
    _network_features = run_agents["network_features"]
    ground_truth = graph_data['config']['statement']['stats']['label_ground_truth']  # type: ignore

    # Class order in belief_ful: [true, false, neither]
    ground_truth_class_idx = 1 if ground_truth == 0  else 0 if ground_truth == 1 else 2
    
    influence_metrics = compute_influence_metrics(belief_ful, A)  # Precompute for all agents
    try:
        agent_ids = [v['node_id'] for k, v in graph_data['agents_data'].items()]
    except:
        agent_ids = [int(k) for k in graph_data['agents_data'].keys()]

    for i in range(belief_ful.shape[0]):
        stats = compute_stats(belief_ful[i], ground_truth_class_idx)  # (N_rounds, 3) per agent
        predicted_label = belief_cat[i][-1]
        final_prob = np.asarray(belief_ful[i][-1], dtype=float)
        
        is_matched = (_models[int(i)], _roles[int(i)]) in MATCHES
        is_expert = not _models[int(i)].startswith("llama-base") 
        recs.append(
            {
                "agent_id": agent_ids[i],
                "plasticity_tv": stats["plasticity_tv"],
                "plasticity_true": stats["plasticity_true"],
                "plasticity_false": stats["plasticity_false"],
                "plasticity_neither": stats["plasticity_neither"],
                "plasticity_tv_1": stats["plasticity_tv_1"],
                "net_change": stats["net_change"],
                "net_gain": stats["net_gain"],
                "progress_ratio": stats["progress_ratio"],
                "progress_forward": stats["progress_forward"],
                "progress_backward": stats["progress_backward"],
                "progress_efficiency": stats["progress_efficiency"],
                "monotonicity": stats["monotonicity"],
                "agent_accuracy": stats["agent_accuracy"],
                "agent_error": stats["agent_error"],
                "T": stats["T"],
                "is_matched": is_matched,
                "is_expert": is_expert,
                "statement_id": run['statement_id'],
                "degree": float(_network_features[int(i)]["degree"]),
                "role": _roles[int(i)],
                "model": _models[int(i)],
                "setting": run["setting"],
                "seed": run["seed"],
                "graph_seed": str(seed_map[str(run["seed"])]),
                "graph_id": f"{run['graph_type']}-{run['seed']}",
                "graph_type": run["graph_type"],
                "centrality": float(_network_features[int(i)]["eigenvector_centrality"]),
                
                "influence_out": influence_metrics["out_influence_mag"][int(i)],
                "influence_in": influence_metrics["in_influence_mag"][int(i)],
                "influence_out_dir": influence_metrics["out_influence_dir"][int(i)],
                "influence_in_dir": influence_metrics["in_influence_dir"][int(i)],
                "effective_n_out": influence_metrics["effective_n_out"][int(i)],
                "effective_n_in": influence_metrics["effective_n_in"][int(i)],
                
                "net_uncertain": stats["diff_uncertain"],
                "monotonicity_uncertainty": stats["monotonicity_uncertainty"],
            }
        )
dft = pd.DataFrame(recs)
dft['setting'] = pd.Categorical(
    dft['setting'], 
    categories=["base_llms", "random_roles", "random_experts", "experts"], 
    ordered=False 
)
dft['model'] = pd.Categorical(dft['model'], 
                              categories=sorted(dft['model'].unique()), 
                              ordered=False)
dft['graph_type'] = pd.Categorical(dft['graph_type'],
                                  categories=sorted(dft['graph_type'].unique()),
                                  ordered=False)
dft['role'] = pd.Categorical(dft['role'],
                                  categories=sorted(dft['role'].unique()),
                                  ordered=False)
dft['statement_id'] = pd.Categorical(dft['statement_id'],
                                  categories=sorted(dft['statement_id'].unique()),
                                  ordered=False)
dft['graph_seed'] = pd.Categorical(dft['graph_seed'],
                                  categories=sorted(dft['graph_seed'].unique()),
                                  ordered=False)

dft['degree_z'] = (dft['degree']- dft['degree'].mean()) / dft['degree'].std()
dft['centrality_z'] = (dft['centrality'] - dft['centrality'].mean()) / dft['centrality'].std()

# Specification of whether agent is a generalist, an expert in a mismatched role, or an expert in a matched role
dft['agent_configs'] = np.select(
    [
        ~dft['is_expert'],
        dft['is_expert'] & ~dft['is_matched'],
        dft['is_expert'] & dft['is_matched'],
    ],
    ['base', 'expert_mismatched', 'expert_matched'],
    default='base'
)

dft['run_id'] = (
    dft['setting'].astype(str) + ':' +
    dft['graph_id'].astype(str) + ':' +
    dft['statement_id'].astype(str)
)
dft['run_id'] = pd.Categorical(dft['run_id'], categories=sorted(dft['run_id'].unique()))


dft.to_parquet(result_path / "agent_level_data.parquet", index=False)


In [11]:
dft.sample(10, random_state=0)

,agent_id,plasticity_tv,plasticity_true,plasticity_false,plasticity_neither,plasticity_tv_1,net_change,net_gain,progress_ratio,progress_forward,...,influence_out_dir,influence_in_dir,effective_n_out,effective_n_in,net_uncertain,monotonicity_uncertainty,degree_z,centrality_z,agent_configs,run_id
37882,10,0.135551,0.093173,0.130762,0.047168,0.141258,0.054445,0.046875,0.035848,0.677246,...,0.163559,-0.050765,24.621018,23.974676,0.054445,0.115428,1.203666,0.117512,expert_matched,experts:erdos-renyi-473392625:false_6
59209,25,0.000104,0.000010,0.000104,0.000094,0.000115,0.000035,0.000035,0.033375,0.000539,...,-0.182109,-0.017695,12.927928,12.561098,0.000035,0.037053,-0.288932,1.588280,expert_mismatched,random_experts:watts-strogatz-900911955:false_9
18712,40,0.005381,0.002980,0.005371,0.002411,0.003156,0.024385,0.024414,0.454545,0.039062,...,0.300738,-0.226259,20.846081,32.635034,0.024385,1.011572,-0.537698,0.222710,base,random_roles:watts-strogatz-814183:false_6
56902,22,0.031043,0.017716,0.028426,0.015943,0.015437,0.140358,0.140358,0.493773,0.212307,...,0.675115,-0.270715,3.430307,11.217251,0.140358,0.880379,-0.537698,1.576313,expert_mismatched,random_experts:watts-strogatz-964669078:false_9
55097,41,0.000106,0.000009,0.000105,0.000098,0.000027,0.000812,0.000812,0.773225,0.000931,...,0.739322,0.019452,3.795598,10.370313,0.000812,0.832746,-0.786464,-0.140115,expert_mismatched,random_experts:watts-strogatz-473392625:false_4
50792,8,0.075230,0.023069,0.070258,0.057133,0.082204,0.070606,0.039986,0.056913,0.371284,...,-0.053107,0.180419,7.727934,9.848032,0.070606,0.123583,-0.786464,0.813685,expert_matched,experts:watts-strogatz-473392625:false_9
58466,2,0.166687,0.126511,0.112699,0.094164,0.168054,0.117261,0.017657,0.013957,0.641382,...,0.139573,0.121376,27.477725,23.052682,0.117261,0.124529,-0.537698,0.926051,expert_mismatched,random_experts:watts-strogatz-1265438423:true_0
9711,15,0.000448,0.000049,0.000397,0.000450,0.000000,0.004476,-0.000488,-1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.004476,0.995545,0.208601,-0.639362,base,base_llms:erdos-renyi-597409993:true_2
56091,27,0.003484,0.000040,0.003472,0.003456,0.001875,0.017253,0.017253,0.496888,0.025988,...,0.062237,0.132528,7.843175,16.322075,0.017253,0.499281,-0.786464,-0.352843,expert_mismatched,random_experts:watts-strogatz-1170252924:false_9
26312,8,0.003545,0.002822,0.003418,0.000851,0.001657,0.018590,-0.017138,-0.607320,0.005540,...,-0.087242,-0.138119,37.574306,53.938885,0.018590,2.185253,1.701199,0.958338,base,random_roles:erdos-renyi-1170252924:true_3


## Processing (Run-based)

In [14]:
import numpy as np

def compute_run_stats(belief_ful: np.ndarray, ground_truth_idx) -> dict:
    """
    Compute consensus metrics across agents within a single run.
    
    Parameters
    ----------
    belief_ful : np.ndarray, shape (N, T+1, 3)
        Per-agent belief trajectories. Last dim is [P(true), P(false), P(neither)].
    
    Returns
    -------
    dict with keys:
        modal_consensus_T       : fraction of agents agreeing with modal label at round T, [1/3, 1]
        modal_consensus_0       : same, at round 0
        modal_consensus_mean    : mean of modal_consensus across all rounds
        polarization_T          : mean pairwise TV distance between agent distributions at T, [0, 1]
                                  high = polarized, low = consensus
        polarization_0          : same, at round 0
        polarization_mean       : mean across rounds
        entropy_T               : Shannon entropy of population label distribution at T (3-class), [0, log(3)]
                                  0 = perfect consensus, log(3) ≈ 1.10 = uniform
        entropy_0               : same, at round 0
        entropy_change          : entropy_T - entropy_0 (positive = more diverse, negative = converged)
        n_distinct_labels_T     : number of distinct modal labels at T (1, 2, or 3)
        majority_label_T        : modal label at round T (0=true, 1=false, 2=neither)
        majority_share_T        : same as modal_consensus_T (alias)
    """
    P = np.asarray(belief_ful, dtype=float)  # (N, T+1, 3)
    N, Tp1, K = P.shape
    T = Tp1 - 1
    
    # Discrete labels per agent per round
    labels = P.argmax(axis=2)  # (N, T+1)
    
    modal_consensus = np.zeros(Tp1)
        
    for t in range(Tp1):
        counts = np.bincount(labels[:, t], minlength=K)
        modal_consensus[t] = counts.max() / N    
    # --- Polarization: mean pairwise TV distance between agents ---
    def mean_pairwise_tv(P_t):
        """P_t: (N, K). Returns scalar."""
        # TV(p, q) = 0.5 * ||p - q||_1
        diffs = np.abs(P_t[:, None, :] - P_t[None, :, :]).sum(axis=2) * 0.5  # (N, N)
        # Exclude diagonal (self-distance = 0)
        n = P_t.shape[0]
        return diffs.sum() / (n * (n - 1))
    
    polarization_T = mean_pairwise_tv(P[:, T])
    polarization_0 = mean_pairwise_tv(P[:, 0])
    polarization_per_round = np.array([mean_pairwise_tv(P[:, t]) for t in range(Tp1)])
    
    accuracy_0 = P[:, 0, ground_truth_idx].mean()
    accuracy_T = P[:, T, ground_truth_idx].mean()
    diff_accuracy = accuracy_T - accuracy_0
    
    # --- Population-level entropy of discrete labels ---
    def label_entropy(labels_t):
        counts = np.bincount(labels_t, minlength=K)
        p = counts / counts.sum()
        # Avoid log(0)
        p_nonzero = p[p > 0]
        return -np.sum(p_nonzero * np.log(p_nonzero))
    
    entropy_T = label_entropy(labels[:, T])
    entropy_0 = label_entropy(labels[:, 0])
    
    # --- Modal label and distinct labels at T ---
    counts_T = np.bincount(labels[:, T], minlength=K)
    majority_label_T = int(counts_T.argmax())
    n_distinct_T = int((counts_T > 0).sum())
    
    return {
        "modal_consensus_T":     float(modal_consensus[T]),
        "modal_consensus_0":     float(modal_consensus[0]),
        "modal_consensus_mean":  float(modal_consensus.mean()),
        "polarization_T":        float(polarization_T),
        "polarization_0":        float(polarization_0),
        "polarization_1":        float(polarization_per_round[1]),
        "polarization_mean":     float(polarization_per_round.mean()),
        "entropy_T":             float(entropy_T),
        "entropy_0":             float(entropy_0),
        "entropy_change":        float(entropy_T - entropy_0),
        "n_distinct_labels_T":   n_distinct_T,
        "majority_label_T":      majority_label_T,
        "majority_share_T":      float(modal_consensus[T]),
        "accuracy_0":            float(accuracy_0),
        "accuracy_T":            float(accuracy_T),
        "diff_accuracy":         float(diff_accuracy),
        "T":                     T,
    }

In [15]:
recs = []

lf = pl.from_pandas(df_runs)

for run in lf.filter(
    pl.col("setting").is_in(["base_llms", "random_roles", "random_experts", "experts"]) 
).iter_rows(named=True):
    
    run_agents = load_agents_data(run["run_path"])
    run_data = load_run_data(run["run_path"])
    A, edges = load_adjacency_matrix(run["run_path"])
    
    net_data = run_data['network_manifest']
    
    
    seed_map = run_data['config']['network']['seed_map']  # type: ignore
    belief_ful = run_agents["belief_ful"]  # (N_agents, N_rounds, 3)
    ground_truth = run_data['config']['statement']['stats']['label_ground_truth']  # type: ignore

    # Class order in belief_ful: [true, false, neither]
    ground_truth_class_idx = 1 if ground_truth == 0  else 0 if ground_truth == 1 else 2
    
    run_stats = compute_run_stats(belief_ful, ground_truth_class_idx)

    recs.append(
        {
                "T": run_stats["T"],
                "statement_id": run['statement_id'],
                "setting": run["setting"],
                "seed": run["seed"],
                "graph_seed": str(seed_map[str(run["seed"])]),
                "graph_id": f"{run['graph_type']}-{run['seed']}",
                "graph_type": run["graph_type"],
                "modal_consensus_T": run_stats["modal_consensus_T"],
                "modal_consensus_0": run_stats["modal_consensus_0"],
                "modal_consensus_mean": run_stats["modal_consensus_mean"],
                "modal_consensus_change": run_stats["modal_consensus_T"] - run_stats["modal_consensus_0"],
                
                "polarization_T": run_stats["polarization_T"],
                "polarization_0": run_stats["polarization_0"],
                "polarization_mean": run_stats["polarization_mean"],
                "polarization_change": run_stats["polarization_T"] - run_stats["polarization_0"],
                'polarization_change_1': run_stats["polarization_T"] - run_stats["polarization_1"],
                "entropy_T": run_stats["entropy_T"],
                "entropy_0": run_stats["entropy_0"],
                "entropy_change": run_stats["entropy_change"],
                "n_distinct_labels_T": run_stats["n_distinct_labels_T"],
                "majority_label_T": run_stats["majority_label_T"],
                "majority_share_T": run_stats["majority_share_T"],
                "ground_truth": run_data['config']['statement']['stats']['label_ground_truth'],
                
                "net_num_nodes": net_data['graph']['num_nodes'],
                "net_num_edges": net_data['graph']['num_edges'],
                "net_radius": net_data['graph']['radius'],
                "net_diameter": net_data['graph']['diameter'],
                "net_degree_mean": net_data['graph']['degrees']['mean'],
                "net_degree_median": net_data['graph']['degrees']['median'],
                "net_degree_std": net_data['graph']['degrees']['std'],
                
                "net_triangles": net_data['graph']['triangles'],
                "net_wedges": net_data['graph']['wedges'],
                "net_avg_path_length": net_data['graph']['avg_path_length'],
                "net_num_components": net_data['graph']['num_components'],
                "net_avg_clustering": net_data['graph']['avg_clustering'],
                "net_global_clustering_coefficient": net_data['graph']['global_clustering_coefficient'],
                "net_density": net_data['graph']['density'],
                
                "accuracy_0": run_stats["accuracy_0"],
                "accuracy_T": run_stats["accuracy_T"],
                "diff_accuracy": run_stats["diff_accuracy"],
        }
    )
dft = pd.DataFrame(recs)
dft['setting'] = pd.Categorical(
    dft['setting'], 
    categories=["base_llms", "random_roles", "random_experts", "experts"], 
    ordered=False 
)

dft['graph_type'] = pd.Categorical(dft['graph_type'],
                                  categories=sorted(dft['graph_type'].unique()),
                                  ordered=False)
dft['statement_id'] = pd.Categorical(dft['statement_id'],
                                  categories=sorted(dft['statement_id'].unique()),
                                  ordered=False)
dft['graph_seed'] = pd.Categorical(dft['graph_seed'],
                                  categories=sorted(dft['graph_seed'].unique()),
                                  ordered=False)

dft['run_id'] = (
    dft['setting'].astype(str) + ':' +
    dft['graph_id'].astype(str) + ':' +
    dft['statement_id'].astype(str)
)
dft['run_id'] = pd.Categorical(dft['run_id'], categories=sorted(dft['run_id'].unique()))


dft.to_parquet(result_path / "run_level_data.parquet", index=False)
